# P23 — GloVe: vectores globales para representación de palabras

## 1. Título y paper

**Paper:** *GloVe: Global Vectors for Word Representation*  
**Autoría:** Jeffrey Pennington, Richard Socher, Christopher D. Manning  
**Año y venue:** 2014 · EMNLP 2014 · ACL Anthology D14-1162  
**Nivel:** L2 · **Motor:** `glove`  
**Ficha completa:** [`P23_glove`](../../papers/foundational/P23_glove/README.md)

**Hito:** Unifica las dos familias de embeddings: factorizar estadísticas globales de co-ocurrencia con la ventaja de los métodos predictivos.

- [ACL Anthology (EMNLP 2014)](https://aclanthology.org/D14-1162/)
- [DOI](https://doi.org/10.3115/v1/D14-1162)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Word2Vec aprendía de ventanas locales y desaprovechaba las estadísticas globales del corpus; los métodos de factorización usaban esas estadísticas pero producían peores analogías.
2. Ejecutar una implementación mínima de la propuesta: Ajustar por mínimos cuadrados ponderados el producto de vectores al logaritmo de la co-ocurrencia, con el argumento de que lo informativo es la RAZÓN de co-ocurrencias, no su valor bruto.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P05
- LSA y la factorización de matrices de co-ocurrencia


## 4. Intuición

Word2Vec mira por una ventanita y aprende de lo que pasa cerca. GloVe cuenta primero todo el corpus y luego busca vectores que expliquen esa tabla de conteos. Mismo destino, camino opuesto.


## 5. Concepto mínimo

```text
J = Σ_ij  f(X_ij) · ( w_i·w̃_j + b_i + b̃_j − log X_ij )²

    X_ij = veces que j aparece en el contexto de i
    f(x) = (x/x_max)^0.75 si x < x_max, si no 1   ← no dejar que los pares frecuentes dominen
```

El argumento del paper no es la fórmula sino **qué se modela**: la RAZÓN `P(k|hielo)/P(k|vapor)` discrimina, y la co-ocurrencia bruta no.


## 6. Código explicado

El motor ajusta la factorización sobre una matriz de co-ocurrencia de juguete con la estructura del ejemplo del paper.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('glove', seed=7)['result']
show(r['razones_de_cooocurrencia'])
print()
for f in r['ajuste_log_cooocurrencia']:
    print(f"{f['par']:<16} log X = {f['log_X']:>6.3f}   predicho = {f['predicho']:>6.3f}")

## 7. Predicción antes de ejecutar

1. ¿Qué razón esperas para «sólido»: mucho mayor que 1, mucho menor, o ≈1?
2. ¿Y para «agua», que acompaña a hielo y a vapor por igual?
3. ¿Por qué haría falta una función de peso f(x)?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('glove', seed=7)['result']
for p in r['perdida']:
    print(f"paso {p['paso']:>3} · pérdida {p['perdida']:.5f}")
print('\nla pérdida baja: los vectores explican cada vez mejor los conteos')

## 9. Salida interpretable

Las tres razones separan limpiamente: ≫1 para lo propio del hielo, ≪1 para lo propio del vapor y ≈1 para lo compartido. **Eso** es lo que los vectores tienen que capturar, y por eso se ajusta al logaritmo: convierte razones en diferencias.


## 10. Comentario pedagógico

Trabajo posterior (Levy y Goldberg, 2015) mostró que word2vec con muestreo negativo factoriza implícitamente una matriz relacionada. La distinción «predictivo frente a contador» resultó ser menos profunda de lo que parecía en 2014: es un buen ejemplo de cómo una dicotomía popular se disuelve.


## 11. Error o anti-patrón deliberado

Anti-patrón: ajustar sin función de peso. Los pares muy frecuentes («de», «la») dominan la pérdida y aplastan a los informativos.


In [ ]:
x_max, alpha = 100.0, 0.75
for x in (1, 5, 50, 100, 5000):
    f = (x / x_max) ** alpha if x < x_max else 1.0
    print(f'X_ij={x:>5} → peso {f:.4f}' + ('   ← saturado, no crece más' if x >= x_max else ''))

## 12. Corrección

Con la función de peso, un par que aparece 5 000 veces no pesa 50× más que uno de 100:


In [ ]:
sin_peso = {x: x for x in (1, 100, 5000)}
con_peso = {x: (min(x, 100) / 100) ** 0.75 for x in (1, 100, 5000)}
print('sin peso :', sin_peso, ' ← el par de 5000 domina la pérdida')
print('con peso :', {k: round(v, 3) for k, v in con_peso.items()})

## 13. Desafío guiado

Comprueba que los pares raros (X_ij pequeño) también aportan poco: la función de peso los atenúa por el otro extremo.


In [ ]:
for x in (1, 2, 3, 10):
    print(f'X_ij={x:>3} → peso {(x / 100) ** 0.75:.4f}  (ruido estadístico, se atenúa)')

## 14. Desafío autónomo

Construye la matriz de co-ocurrencia de un corpus público pequeño, entrena GloVe y word2vec con la misma dimensión, y compara ambos en un conjunto de analogías propio. Reporta también el tiempo de entrenamiento: es el argumento práctico que más pesó en su momento.


## 15. Evidencia de aprendizaje

Guarda la tabla de razones, el ajuste log X frente a predicho, y tu explicación de por qué se modela el logaritmo y no la co-ocurrencia directa.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P23_glove/README.md) · evaluación formal: [`assessments/papers/P23_glove.md`](../../assessments/papers/P23_glove.md)


## 16. Cierre

Las palabras ya tienen geometría global. Pero siguen teniendo **un solo vector por palabra**, y eso sigue sin resolver la polisemia.


## 17. Conexión con el siguiente hito

- P24
- P18

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
